# Reproducing Geiller, Vancura et al. (2020) from dandiset 000625

Paper: Geiller T, Vancura B, Terada S, et al. (2020). Large-Scale 3D Two-Photon Imaging of
Molecularly Identified CA1 Interneuron Dynamics in Behaving Mice. Neuron 108(5):968-983.
[doi:10.1016/j.neuron.2020.09.013](https://doi.org/10.1016/j.neuron.2020.09.013). The full text
was used: the NIH author manuscript on PubMed Central (PMC7736348), with its STAR Methods and
figure legends. The supplementary figures were not available, so claims about them rest on the
Results text.

Data: [DANDI:000625](https://dandiarchive.org/dandiset/000625/0.231114.0019), version
0.231114.0019, streamed from the archive. The paper cites no version of the dandiset (it says the
data "will be made available soon"), so the most recent published version is pinned. The only
other published version, 0.231114.0017, has the same three assets; the two differ only in the
description text (checked below).

## What can and cannot be tested

The paper imaged 3626 interneurons in 16 mice across random foraging, novel contexts,
goal-oriented learning, sharp-wave ripple recordings and two further staining strategies. The
dandiset holds one VGAT-Cre mouse (tg19) in three random-foraging sessions: dF/F of 231 ROIs, the
subtype of each ROI, belt position, and reward and water times. It holds no cell depth, LFP,
licking, novel-context or goal-oriented sessions, and none of the VIP-Cre or strategy 3 mice.

| Claim | Paper | Here |
|---|---|---|
| `fig1-cells-per-mouse` | About 227 interneurons imaged and 121 identified per mouse | Tested |
| `fig1-depth-distribution` | Layer distribution of each subtype | Blocked: no cell depth or coordinates |
| `fig2c-most-positive` | Most interneurons positively correlated with velocity, about 10% negatively | Tested |
| `fig2c-cck-negative` | Negative correlations enriched in CCKC, nearly half of them | Tested |
| `fig2c-subtypes-differ` | Velocity correlation differs between subtypes | Tested |
| `fig2c-ngfc-less-modulated` | IvC/NGFC less modulated by locomotion | Tested |
| `fig2d-negative-cells-in-oriens` | Negatively correlated cells in stratum oriens | Blocked: no cell depth |
| `s2-lags` | Negatively correlated cells more tightly locked in time to velocity | Tested |
| `s2-cck-run-stop` | Immobility-triggered activation specific to CCKC | Tested |
| `s3-subtype-classification` | Deep network classifies four groups at 59 to 71% | Blocked: one mouse of the training set, no depth |
| `fig3-swr-pvbc-vs-somc`, `fig3-swr-bimodal` | Sharp-wave ripple modulation | Blocked: no LFP or 40 Hz sessions |
| `fig4-fraction-spatial` | 17.5% of interneurons spatially selective | Tested |
| `fig4-fraction-negative` | 12.7% with negative fields | Tested |
| `fig4d-decoding` | Spatially selective subset decodes position about as well as all cells | Tested |
| `fig4e-bic-recurrence` | BiC more often spatially modulated again | Blocked: 3 BiC |
| `fig4f-selectivity-no-difference` | Selectivity index does not differ between subtypes | Tested |
| `fig4h-pvbc-more-stable` | Spatial PVBC more stable within session than spatial SomC | Tested |
| `s6-within-session-order` | PVBC most and CCKC least stable within session | Tested |
| `s6-later-lower-above-chance` | Stability after 1 h and 24 h lower but above chance | Tested, with stand-in intervals |
| `fig5-context-decorrelation`, `fig5-novel-context-decrease` | Novel context | Blocked: no novel-context sessions |
| `fig6-reward-somc-bic`, `fig6-old-zone` | Goal-oriented learning | Blocked: no goal-oriented sessions or licking |
| `fig7-vip-subtypes-immobility`, `fig7-isi-correlations` | VIP and ISI subtypes | Blocked: those mice are not in the dandiset |

The paper's code ([losonczylab/Geiller_Vancura_Neuron2020](https://github.com/losonczylab/Geiller_Vancura_Neuron2020),
commit c37f37c) holds only the deep-network classifier of Figure S3 and its preprocessing. From it
this notebook takes the lap definition (a lap starts where position wraps to the start of the
belt). Everything else follows the STAR Methods, with the choices they leave open stated in
`interneurons.py`.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from threadpoolctl import threadpool_limits

import interneurons as inn

warnings.filterwarnings("ignore", category=RuntimeWarning)
threadpool_limits(1)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.precision", 3)
pd.set_option("display.width", 160)


def rng_for(claim):
    """A generator of its own for each claim's section, seeded from the claim id."""
    return np.random.default_rng(list(claim.encode()))

## Versions

The two published versions list the same assets, with the same asset identifiers.

In [ ]:
old, new = inn.assets("0.231114.0017"), inn.assets(inn.VERSION)
print(pd.DataFrame(new, columns=["path", "asset id"]).to_string(index=False))
print(f"\nThe assets of 0.231114.0017 and {inn.VERSION} are the same: {old == new}")

## Load the data and check it against its description

`interneurons.load_session` streams each file and reads the dF/F traces, the subtype table, the
belt position, the reward and water intervals and the ROI masks. The raw image series are not
read.

In [ ]:
sessions = {s.session_id: s for s in (inn.load_session(p) for p, _ in new)}
order = sorted(sessions, key=lambda k: sessions[k].start_time)      # chronological
rows = []
for sid in order:
    s = sessions[sid]
    L = inn.belt_length(s)
    finite = s.dff[np.isfinite(s.dff)]
    rows.append({
        "session": sid, "start (as stored)": s.start_time, "description": s.description,
        "frames": s.dff.shape[0], "ROIs": s.dff.shape[1], "ROIs imaged": int((~np.isnan(s.dff).all(0)).sum()),
        "ROIs partly NaN": int((np.isnan(s.dff).any(0) & ~np.isnan(s.dff).all(0)).sum()),
        "stated rate (Hz)": s.stated_rate, "imaging s at stated rate": s.dff.shape[0] / s.stated_rate,
        "behaviour s": s.pos_t[-1], "position samples": len(s.pos),
        "longest gap between samples (s)": np.diff(s.pos_t).max(),
        "position min (m)": s.pos.min(), "position max (m)": s.pos.max(), "belt length (m)": L,
        "laps (wraps)": int((np.diff(s.pos) < -L / 2).sum()),
        "reward intervals": len(s.reward), "median reward s": np.median(np.diff(s.reward, axis=1)),
        "water intervals": len(s.water), "median water s": np.median(np.diff(s.water, axis=1)),
        "dF/F min": finite.min(), "dF/F median": np.median(finite), "dF/F 99th pct": np.percentile(finite, 99),
        "dF/F max": finite.max(),
    })
print(pd.DataFrame(rows).set_index("session").T.to_string())

first = sessions[order[0]]
print("\nLabels as stored, ROIs per label (the same in every session: "
      f"{all((sessions[k].label == first.label).all() for k in order)})")
print(pd.Series(first.label).value_counts().to_string())
print("\nThe paper's six subtypes, ROIs imaged per session")
print(pd.DataFrame({sid: pd.Series(sessions[sid].subtype[~np.isnan(sessions[sid].dff).all(0)])
                    .value_counts().reindex(inn.ORDER) for sid in order}).to_string())
print(f"\nMetadata read from the files, session {order[0]}")
for k, v in first.meta.items():
    print(f"  {k}: {v}")
for sid in order[1:]:
    diff = {k: v for k, v in sessions[sid].meta.items() if v != first.meta[k]}
    print(f"Session {sid} differs in: {diff if diff else 'nothing'}")

Checks of the description against the values:

- Subject and sessions. One mouse, tg19, which the paper lists under staining strategy 1. Three
  sessions, two on 2019-06-13 about 2.5 h apart and one two days earlier. All three are described
  as "session 1 of 4", and the experiment description mentions four random-foraging sessions; one
  is not in the dandiset.
- ROIs and subtypes. 231 ROIs, of which 215, 216 and 224 have traces in the three sessions; the
  others are entirely NaN. The subtype table has 232 rows; the row with id 231 has no ROI and is
  left out, and the others are matched to ROIs by id. 74 ROIs carry one of the paper's six
  subtypes (24 PVBC, 10 SOM, 3 BISTR, 9 AAC counting "AAC (RIGHT CELL)", 15 CCK, 13 NGFF). The
  labels "UNIDENTIFIED" and "NOTHING" are not explained. The column's description reads
  "Recording location in Middle Earth".
- dF/F. The values run from about -0.15 to 8.8 with a median near 0.12, so they are fractions,
  not percent. The stated unit, lumens, is wrong. The DfOverF and Fluorescence series are the same
  HDF5 object, so the raw fluorescence the description lists is not there.
- Position. Stated in metres, it runs from 0 to 2.11 m, so the belt is 2.11 m long here, where the
  paper says 2 m. Position is stored only when it changes (gaps of up to 66 s), so the value at a
  frame is the last stored sample.
- Imaging rate. Every file states 5.7981 Hz. For session 21333 that gives 899 s of imaging against
  897 s of behaviour. For 21324 and 21322 it gives 680 s against 900 s and 879 s. This is checked
  next.
- Raw images. One cropped series per file (33 x 46 or 25 x 46 pixels), where the description says
  "individual cropped ROIs". The emission wavelength is given as 920 nm, the same as the
  excitation. The subject's age is P60D in all three sessions, where the paper gives 3 to 5
  months.

### Do imaging and behaviour share a clock?

The files give no frame timestamps, only a rate. For each session, `fit_frame_rate` places the
frames at i / rate for rates from 3.5 to 6.5 Hz and scores each rate by the best correlation (over
lags within 5 s) between the mean z-scored dF/F and running speed. Then, as a contrast, running
(speed above 5 cm/s) is decoded from the population with the labels shifted in time, on the stated
and on the fitted clock. With the right clock, decoding peaks near zero shift and falls to chance
far from it.

In [ ]:
rates = np.arange(3.5, 6.5, 0.005)
clock, scores, rows = {}, {}, []
for sid in order:
    s = sessions[sid]
    clock[sid], scores[sid] = inn.fit_frame_rate(s, rates)
    rows.append({"session": sid, "stated rate": s.stated_rate, "fitted rate": clock[sid],
                 "score at stated rate": scores[sid][np.argmin(abs(rates - s.stated_rate))],
                 "score at fitted rate": scores[sid].max(),
                 "imaging s at fitted rate": s.dff.shape[0] / clock[sid], "behaviour s": s.pos_t[-1]})
print(pd.DataFrame(rows).set_index("session").to_string())
fig, ax = plt.subplots(figsize=(6, 3))
for sid in order:
    ax.plot(rates, scores[sid], label=sid)
ax.axvline(5.7981, color="k", ls="--", lw=1, label="stated")
ax.set_xlabel("Frame rate (Hz)"); ax.set_ylabel("Best r, population vs speed"); ax.legend(fontsize=8)
print(f"Plotted scores range from {min(v.min() for v in scores.values()):.2f} to {max(v.max() for v in scores.values()):.2f}")

In [ ]:
shifts = np.arange(-60, 61, 5.0)
rows, curves = [], {}
for sid in order:
    s = sessions[sid]
    for name, rate in (("stated", s.stated_rate), ("fitted", clock[sid])):
        b = inn.behaviour(s, rate)
        acc = inn.shifted_label_decoding(s.dff, b["speed"] > inn.RUN_SPEED, b["covered"], rate, shifts)
        curves[sid, name] = acc
        rows.append({"session": sid, "clock": name, "rate": rate, "accuracy at 0 s": acc[shifts == 0][0],
                     "best shift (s)": shifts[np.argmax(acc)], "best accuracy": acc.max(),
                     "mean accuracy at |shift| >= 30 s": acc[abs(shifts) >= 30].mean()})
print("Balanced accuracy of decoding running (> 5 cm/s) from the population; chance is 0.5")
print(pd.DataFrame(rows).to_string(index=False))
fig, axes = plt.subplots(1, 3, figsize=(10, 2.8), sharey=True)
for ax, sid in zip(axes, order):
    for name in ("stated", "fitted"):
        ax.plot(shifts, curves[sid, name], marker=".", label=name)
    ax.axhline(0.5, color="0.7", lw=1); ax.set_title(sid, fontsize=10); ax.set_xlabel("Label shift (s)")
axes[0].set_ylabel("Balanced accuracy"); axes[0].legend(fontsize=8)
print(f"Plotted accuracies range from {min(c.min() for c in curves.values()):.2f} to {max(c.max() for c in curves.values()):.2f}")

For 21333 the fit recovers the stated rate (5.800 against 5.798 Hz). For 21324 and 21322 the
stated rate gives a population-speed correlation of 0.16 and 0.09 and decoding at chance at zero
shift (0.51 and 0.44), while the fitted rates, 4.395 and 4.385 Hz, give 0.72 and 0.87 and
decoding of 0.94 and 0.89 at zero shift, falling to chance (0.49 and 0.50) beyond 30 s. The
fitted rates also make all three sessions about 898 s long (898, 898 and 899 s), as session
21333's stated rate does, which is evidence independent of the neural data. The data are trusted:
the rest of the notebook uses the fitted rates, with frames starting at time 0 as the files
state. The fit uses the same population-speed relation that some claims test, so the correlation
of the population with speed is in part fitted; the stated clock is shown as a variant where it
matters. In 21322, imaging continues about 20 s after the position record ends; those frames are
left out.

In [ ]:
# Behaviour on the fitted clock, and each ROI's velocity correlation and tuning curve, per session.
B, RUN, BINS, TC, R, LAG, IMAGED = {}, {}, {}, {}, {}, {}, {}
for sid in order:
    s = sessions[sid]
    b = B[sid] = inn.behaviour(s, clock[sid])
    IMAGED[sid] = ~np.isnan(s.dff).all(0)
    R[sid], LAG[sid] = inn.velocity_correlation(s.dff, b["speed"], clock[sid], b["covered"])
    RUN[sid] = b["covered"] & (b["speed"] > inn.RUN_SPEED)
    BINS[sid] = inn.position_bins(b["pos"][RUN[sid]], b["length"])
    TC[sid] = inn.tuning_curves(s.dff[RUN[sid]], BINS[sid])
    TC[sid][~IMAGED[sid]] = np.nan
print(pd.DataFrame({sid: {"running frames": RUN[sid].sum(), "fraction running": RUN[sid][B[sid]["covered"]].mean(),
                          "laps": B[sid]["lap"].max(), "speed max (m/s)": B[sid]["speed"].max(),
                          "speed min (m/s)": B[sid]["speed"].min()} for sid in order}).to_string())

### Repeated deposits and cell identity across sessions

The three files are checked for duplicated recordings (identical traces or behaviour), and for
whether the ROI with the same index is the same cell in every session, as the dandiset's
description implies ("231 ROIs were observed over 3 separate imaging sessions") and as the
paper's averaging over sessions requires. The tests compare, for the same ROI in two sessions,
mask area, velocity correlation and tuning curve, against the same quantity with ROIs shuffled.

In [ ]:
rng = rng_for("identity")
pairs = [(a, b) for i, a in enumerate(order) for b in order[i + 1:]]
rows = []
for a, b in pairs:
    both = IMAGED[a] & IMAGED[b]
    n = min(len(sessions[a].dff), len(sessions[b].dff))
    same_trace = inn.rowwise_corr(sessions[a].dff[:n, both].T, sessions[b].dff[:n, both].T)
    pos_r = np.corrcoef(B[a]["pos"][:n], B[b]["pos"][:n])[0, 1]
    ma, mb = sessions[a].mask_area[both], sessions[b].mask_area[both]
    mask_r = np.corrcoef(ma, mb)[0, 1]
    mask_null = [np.corrcoef(ma, rng.permutation(mb))[0, 1] for _ in range(1000)]
    ta, tb = TC[a][both], TC[b][both]
    tune = np.nanmean(inn.rowwise_corr(ta, tb))
    tune_null = [np.nanmean(inn.rowwise_corr(ta, tb[rng.permutation(len(tb))])) for _ in range(200)]
    vel_r = np.corrcoef(R[a][both], R[b][both])[0, 1]
    vel_null = [np.corrcoef(R[a][both], rng.permutation(R[b][both]))[0, 1] for _ in range(1000)]
    rows.append({"pair": f"{a}-{b}", "ROIs in both": both.sum(),
                 "max |r| same ROI, raw traces": np.nanmax(abs(same_trace)), "r of positions": pos_r,
                 "mask area r": mask_r, "mask area null 95th": np.percentile(mask_null, 95),
                 "velocity r, r": vel_r, "velocity r null 95th": np.percentile(vel_null, 95),
                 "mean tuning r same ROI": tune, "tuning null mean": np.mean(tune_null),
                 "tuning null 95th": np.percentile(tune_null, 95)})
print(pd.DataFrame(rows).set_index("pair").T.to_string())

No two sessions duplicate each other: the largest correlation of a ROI's trace with itself in
another session is 0.22, and the position traces correlate at 0.11 or less. Identity across
sessions is consistent with the data but not shown. The velocity correlation of the same ROI
agrees across sessions above the shuffled 95th percentile in all three pairs (0.16, 0.16 and 0.41
against about 0.11). Mask areas agree above it in two pairs of three (0.13 and 0.21 against 0.12
and 0.11, and -0.05 for 21333-21322). Tuning curves agree above it only for 21333-21324 (0.035
against 0.026). The ROI index is taken as the cell, as the dandiset describes; claims that
average over sessions are also shown per session.

## Cells per mouse (`fig1-cells-per-mouse`)

> About 227 interneurons were imaged per mouse (226.6 ± 27.3, mean ± std over 16 mice), and
> 121.0 ± 28.4 per mouse were assigned a putative subtype.

In [ ]:
s0 = sessions[order[0]]
imaged_any = np.any([IMAGED[sid] for sid in order], axis=0)
table = pd.DataFrame({
    "ROIs in the files": [len(s0.roi_ids)],
    "ROIs imaged in at least one session": [imaged_any.sum()],
    "in the six subtypes": [(s0.subtype != "").sum()],
    "not labelled UNIDENTIFIED": [(s0.label != "UNIDENTIFIED").sum()],
}, index=["mouse tg19"])
print(table.T.to_string())
print("Paper: 226.6 +/- 27.3 imaged and 121.0 +/- 28.4 assigned per mouse (mean +/- std, 16 mice)")
print(f"Ranges: imaged {226.6 - 27.3:.1f} to {226.6 + 27.3:.1f}, assigned {121.0 - 28.4:.1f} to {121.0 + 28.4:.1f}")

**Result: partially reproduced.** The 231 ROIs of mouse tg19 are within the paper's spread of
226.6 ± 27.3. The ROIs assigned a subtype, 74 in the six subtypes or 83 including "NOTHING", are
below the range of 121.0 ± 28.4 (92.6 to 149.4) under both readings.

## Correlation with velocity (`fig2c-most-positive`)

> The majority of interneurons were positively correlated with running velocity, and about 10%
> were negatively correlated, across all subtypes.

Each ROI's correlation is the signed Pearson r of largest absolute value over lags from -5 to
+5 s, averaged over the sessions in which it was imaged, as in the paper's methods.

In [ ]:
cells = pd.DataFrame({"subtype": s0.subtype, "label": s0.label,
                      "r": np.nanmean([R[sid] for sid in order], axis=0),
                      "lag": np.nanmean([LAG[sid] for sid in order], axis=0),
                      "sessions": np.sum([IMAGED[sid] for sid in order], axis=0)})
cells = cells[cells.sessions > 0]
six = cells[cells.subtype != ""]


def summary(r):
    return {"n": len(r), "fraction positive": np.mean(r > 0), "fraction negative": np.mean(r < 0),
            "median r": np.median(r)}


rows = {"all imaged ROIs": summary(cells.r), "six subtypes": summary(six.r)}
for sid in order:
    rows[f"session {sid}, all"] = summary(R[sid][IMAGED[sid]])
    m = IMAGED[sid] & (s0.subtype != "")
    rows[f"session {sid}, six subtypes"] = summary(R[sid][m])
print(pd.DataFrame(rows).T.to_string())
print(f"Paper: 156 of 1585 negative ({156 / 1585:.3f}); with a tolerance of 10%, {0.098 * 0.9:.3f} to {0.098 * 1.1:.3f}")

In [ ]:
# Variants: the smoothing of velocity, and the stated clock.
rows = {}
for label, sigma, use_stated in (("velocity unsmoothed", 0.0, False), ("velocity smoothed 0.5 s (main)", 0.5, False),
                                 ("velocity smoothed 1 s", 1.0, False), ("stated clock, 0.5 s", 0.5, True)):
    rs = []
    for sid in order:
        rate = sessions[sid].stated_rate if use_stated else clock[sid]
        b = inn.behaviour(sessions[sid], rate, sigma)
        rs.append(inn.velocity_correlation(sessions[sid].dff, b["speed"], rate, b["covered"])[0])
    r = np.nanmean(rs, axis=0)
    r = r[~np.isnan(r)]
    rows[label] = {**summary(r), **{f"negative in {sid}": np.mean(x[~np.isnan(x)] < 0) for sid, x in zip(order, rs)}}
print(pd.DataFrame(rows).T.to_string())
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(cells.r, bins=np.linspace(-1, 1, 41), color="0.4")
ax.axvline(0, color="k", lw=1); ax.set_xlabel("Correlation with velocity"); ax.set_ylabel("ROIs")
print(f"Plotted r ranges from {cells.r.min():.2f} to {cells.r.max():.2f}")

**Result: reproduced.** Over the sessions, 96.5% of the 228 imaged ROIs and 90.5% of the 74 ROIs
in the six subtypes are positively correlated with velocity. The fraction negative is 0.095 among
the six subtypes, inside the tolerance of 0.088 to 0.108, and 0.035 among all imaged ROIs,
outside it. Per session the fraction negative among all ROIs is 0.065, 0.111 and 0.089. Smoothing
velocity over 0 or 1 s changes no fraction by more than 0.005. On the stated clock, 28% of ROIs in
session 21322 are negatively correlated.

## Negative correlations in CCKC (`fig2c-cck-negative`)

> Negatively correlated cells were significantly enriched in CCKC, where nearly half of cells
> were more active during immobility.

In [ ]:
by_type = six.groupby("subtype").r.agg(n="count", mean="mean", median="median",
                                       fraction_negative=lambda x: np.mean(x < 0)).reindex(inn.ORDER)
print(by_type.to_string())
per_session = pd.DataFrame({sid: pd.Series(R[sid][IMAGED[sid] & (s0.subtype != "")],
                                           index=s0.subtype[IMAGED[sid] & (s0.subtype != "")])
                            .groupby(level=0).apply(lambda x: np.mean(x < 0)).reindex(inn.ORDER) for sid in order})
print("\nFraction negative per session")
print(per_session.to_string())
cck = six.subtype == "CCKC"
table = [[(six.r[cck] < 0).sum(), (six.r[cck] >= 0).sum()], [(six.r[~cck] < 0).sum(), (six.r[~cck] >= 0).sum()]]
odds, p = stats.fisher_exact(table, alternative="greater")
print(f"\nCCKC negative {table[0][0]} of {sum(table[0])}; other five subtypes {table[1][0]} of {sum(table[1])}; "
      f"one-sided Fisher exact p = {p:.2g}, odds ratio {odds:.1f}")

**Result: reproduced.** 6 of 15 CCKC (0.40) are negatively correlated, against 1 of 59 cells in
the other five subtypes (one-sided Fisher exact p = 0.00017). By session the CCKC fraction is
0.13, 0.50 and 0.43.

## Differences between subtypes (`fig2c-subtypes-differ`)

> The correlation with velocity differs between the six subtypes (one-way ANOVA, p < 1e-10).

In [ ]:
groups = [six.r[six.subtype == t].to_numpy() for t in inn.ORDER]
F, p = stats.f_oneway(*groups)
print(f"One-way ANOVA across the six subtypes: F = {F:.2f}, p = {p:.2g} (n = {[len(g) for g in groups]})")
for sid in order:
    m = IMAGED[sid] & (s0.subtype != "")
    g = [R[sid][m & (s0.subtype == t)] for t in inn.ORDER]
    print(f"  session {sid}: F = {stats.f_oneway(*g)[0]:.2f}, p = {stats.f_oneway(*g)[1]:.2g}")
tk = stats.tukey_hsd(*groups)
rows = [{"pair": f"{inn.ORDER[i]} - {inn.ORDER[j]}", "difference": tk.statistic[i, j], "p": tk.pvalue[i, j]}
        for i in range(6) for j in range(i + 1, 6)]
print("\nTukey's test")
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** The one-way ANOVA across the six subtypes gives p = 0.0015. By Tukey's
test the difference lies with CCKC, lower than PVBC (p = 0.0009) and IvC/NGFC (p = 0.005). Within
single sessions the ANOVA is significant in one of three (21324, p = 1.1e-05).

## IvC/NGFC and locomotion (`fig2c-ngfc-less-modulated`)

> IvC/NGFC were less modulated by locomotion than other subtypes.

In [ ]:
ngfc = six.r[six.subtype == "IvC/NGFC"]
others = six.r[six.subtype.isin(["PVBC", "SomC", "BiC", "AAC"])]
rows = []
for name, f in (("r", lambda x: x), ("|r|", np.abs)):
    p = stats.mannwhitneyu(f(ngfc), f(others), alternative="less").pvalue
    rows.append({"reading": name, "IvC/NGFC mean": f(ngfc).mean(), "IvC/NGFC n": len(ngfc),
                 "PVBC, SomC, BiC, AAC mean": f(others).mean(), "n": len(others), "one-sided Mann-Whitney p": p})
    for sid in order:
        m = IMAGED[sid]
        a = f(R[sid][m & (s0.subtype == "IvC/NGFC")])
        o = f(R[sid][m & np.isin(s0.subtype, ["PVBC", "SomC", "BiC", "AAC"])])
        rows.append({"reading": f"{name}, session {sid}", "IvC/NGFC mean": a.mean(), "IvC/NGFC n": len(a),
                     "PVBC, SomC, BiC, AAC mean": o.mean(), "n": len(o),
                     "one-sided Mann-Whitney p": stats.mannwhitneyu(a, o, alternative="less").pvalue})
print(pd.DataFrame(rows).to_string(index=False))

**Result: not reproduced.** Averaged over sessions, IvC/NGFC have a mean correlation of 0.511
against 0.484 for PVBC, SomC, BiC and AAC (one-sided p = 0.58), and the same holds for the
absolute correlation. Only session 21324 shows the effect (0.368 against 0.525, p = 0.005).

## Lags (`s2-lags`)

> The activity of positively correlated cells followed changes in velocity by up to several
> seconds, while the activity of negatively correlated cells was more tightly linked in time.

In [ ]:
def lag_test(r, lag):
    pos, neg = lag[r > 0], lag[r < 0]
    return {"positive n": len(pos), "positive median lag (s)": np.median(pos),
            "positive median |lag| (s)": np.median(abs(pos)), "negative n": len(neg),
            "negative median lag (s)": np.median(neg), "negative median |lag| (s)": np.median(abs(neg)),
            "one-sided Mann-Whitney p (|lag| neg < pos)": stats.mannwhitneyu(abs(neg), abs(pos), alternative="less").pvalue}


rows = {"all imaged ROIs, averaged over sessions": lag_test(cells.r.to_numpy(), cells.lag.to_numpy()),
        "six subtypes, averaged over sessions": lag_test(six.r.to_numpy(), six.lag.to_numpy())}
for sid in order:
    rows[f"session {sid}"] = lag_test(R[sid][IMAGED[sid]], LAG[sid][IMAGED[sid]])
print(pd.DataFrame(rows).T.to_string())
print("Negative lag: activity lags velocity.")
fig, ax = plt.subplots(figsize=(5, 3))
ax.scatter(cells.lag, cells.r, s=10, color="0.3")
ax.axhline(0, color="0.7", lw=1); ax.axvline(0, color="0.7", lw=1)
ax.set_xlabel("Lag of peak correlation (s)"); ax.set_ylabel("Correlation with velocity")
print(f"Plotted lags range from {cells.lag.min():.2f} to {cells.lag.max():.2f} s")

**Result: not reproduced.** Positively correlated ROIs lag velocity, with a median lag of -1.67 s,
as the paper says. The 8 negatively correlated ROIs have a larger median absolute lag (2.06 s),
not a smaller one (one-sided p = 0.60), and so do the negatively correlated ROIs of each session
taken alone. The lags depend on frames starting at time 0, which the files state but which could
not be checked independently.

## Run-stop responses (`s2-cck-run-stop`)

> Run-start and run-stop responses show that immobility-triggered activation is specific to
> CCKC.

Events are crossings of 0.2 cm/s separated by at least 5 s from the previous event; the response
is the mean dF/F in the 3 s after the event minus the 3 s before.

In [ ]:
def run_responses(separation):
    start, stop, counts = [], [], {}
    for sid in order:
        s, b = sessions[sid], B[sid]
        ev = inn.run_events(b["speed"], clock[sid], b["covered"], separation)
        counts[sid] = {k: len(v) for k, v in ev.items()}
        start.append(inn.event_response(s.dff, ev["start"], clock[sid]))
        stop.append(inn.event_response(s.dff, ev["stop"], clock[sid]))
    return np.nanmean(start, axis=0), np.nanmean(stop, axis=0), counts


rows, events = [], {}
for sep in (3.0, 5.0, 10.0):
    start, stop, events[sep] = run_responses(sep)
    df = pd.DataFrame({"subtype": s0.subtype, "start": start, "stop": stop}).loc[cells.index]
    df = df[df.subtype != ""]
    c, o = df.stop[df.subtype == "CCKC"].dropna(), df.stop[df.subtype != "CCKC"].dropna()
    means = df.groupby("subtype")[["start", "stop"]].mean().reindex(inn.ORDER)
    rows.append({"separation (s)": sep, **{f"stop {t}": means.loc[t, "stop"] for t in inn.ORDER},
                 "CCKC n": len(c), "others n": len(o),
                 "one-sided Mann-Whitney p (CCKC stop > others)": stats.mannwhitneyu(c, o, alternative="greater").pvalue})
    if sep == 5.0:
        main = means
print("Events kept per session:", events)
print("\nMean run-start and run-stop responses (dF/F, 3 s after minus 3 s before), separation 5 s")
print(main.to_string())
start, stop, _ = run_responses(5.0)
neg_cck = cells.index[(cells.subtype == "CCKC") & (cells.r < 0)]
pos_cck = cells.index[(cells.subtype == "CCKC") & (cells.r > 0)]
print(f"\nCCKC run-stop response (separation 5 s): negatively correlated ({len(neg_cck)}) "
      f"{np.nanmean(stop[neg_cck]):.3f}, positively correlated ({len(pos_cck)}) {np.nanmean(stop[pos_cck]):.3f}")
print("\nRun-stop responses by separation")
print(pd.DataFrame(rows).set_index("separation (s)").T.to_string())

**Result: partially reproduced.** At run stops every other subtype drops by 0.06 to 0.10 dF/F,
while CCKC stay flat on average (-0.003), a significant difference (one-sided p = 0.0009, the
same at separations of 3 and 10 s). The criterion also asks that CCKC rise on average, and they do
not. Only the 6 negatively correlated CCKC rise (0.084, against -0.061 for the other 9; shown
above, not part of the criterion).

## Spatially selective interneurons (`fig4-fraction-spatial`)

> 17.5 ± 6.4% of interneurons, blind of subtype, passed the test for significant spatial
> selectivity.

The paper's rule, on frames with speed above 5 cm/s: tuning curves of 100 bins smoothed with a
Gaussian of 3 bins; 1000 shuffles rotating position against dF/F; a field is 10 consecutive bins
above the shuffles' 95th percentile, with a peak above 100% dF/F. Fields may wrap around the end
of the belt. As a null, the same procedure is run once per session on a surrogate in which dF/F
is first rotated against the behaviour by at least 60 s.

In [ ]:
rng = rng_for("fig4-fraction-spatial")
BOUNDS, rows = {}, []
for sid in order:
    s, m = sessions[sid], IMAGED[sid]
    BOUNDS[sid] = inn.shuffle_bounds(s.dff[RUN[sid]][:, m], BINS[sid], rng)
    # The same procedure on a surrogate: dF/F rotated against the behaviour by at least 60 s first.
    shift = rng.integers(int(60 * clock[sid]), RUN[sid].sum() - int(60 * clock[sid]))
    surrogate = inn.shuffle_bounds(np.roll(s.dff[RUN[sid]][:, m], shift, axis=0), BINS[sid], rng)
    for name, kw in (("peak > 100% dF/F", {}), ("no peak rule", {"peak": None}),
                     ("peak > 50% dF/F", {"peak": 0.5}), ("no peak rule, no wrap", {"peak": None, "wrap": False}),
                     ("no peak rule, 15 bins", {"peak": None, "field_bins": 15})):
        f, fs = inn.place_fields(BOUNDS[sid], **kw), inn.place_fields(surrogate, **kw)
        rows.append({"session": sid, "rule": name, "ROIs": m.sum(), "positive": f["positive"].mean(),
                     "negative": f["negative"].mean(), "positive, surrogate": fs["positive"].mean(),
                     "negative, surrogate": fs["negative"].mean()})
spatial = pd.DataFrame(rows)
print(spatial.to_string(index=False))
print("\nMean over the three sessions")
print(spatial.groupby("rule", sort=False)[["positive", "negative", "positive, surrogate", "negative, surrogate"]].mean().to_string())
print("Paper: positive 0.175 +/- 0.064, negative 0.127 +/- 0.083 (mean +/- sem)")
peaks = np.concatenate([np.nanmax(BOUNDS[sid]["tc"], 1) for sid in order])
print(f"\nPeak of the session tuning curve per ROI: median {np.median(peaks):.2f}, 90th percentile "
      f"{np.percentile(peaks, 90):.2f}, max {peaks.max():.2f} dF/F; above 1.0 in {np.mean(peaks > 1):.3f} of ROIs")

**Result: not reproduced.** With the paper's peak rule, 1.0% of ROIs are selected on average over
the sessions, because only 1.1% of ROIs have a tuning curve that reaches 1.0 dF/F. Without the
peak rule, 69.8% are selected, against 8.6% on the surrogate. Neither is within 17.5 ± 6.4%. A
peak rule of 50% dF/F gives 13.0%, inside the range, but that threshold is not the paper's.

In [ ]:
print(spatial[spatial.rule == "no peak rule"][["session", "negative", "negative, surrogate"]].to_string(index=False))
print(f"Mean over sessions: {spatial[spatial.rule == 'no peak rule'].negative.mean():.3f}")

**Result: not reproduced.** On average 66.4% of ROIs have a negative field (47.0%, 74.5% and
77.7% by session), against 10.8% on the surrogate and 12.7 ± 8.3% in the paper.

## Position decoding (`fig4d-decoding`)

> Position decoded from the small subset of spatially selective interneurons was about as
> accurate as from all interneurons, and more accurate than from as many randomly chosen
> interneurons or with position shuffled.

A linear SVM decodes the position bin from lap-wise tuning curves, leaving one lap out. The
selected cells follow the peak rule where it gives at least 5 cells (only in 21322) and otherwise
the rule without the peak. Shuffled position is dF/F rotated against position by at least 60 s
(three draws); random subsets are ten draws of the same size.

In [ ]:
rng = rng_for("fig4d-decoding")
SEL, rows = {}, []
for sid in order:
    s, m = sessions[sid], IMAGED[sid]
    peak = inn.place_fields(BOUNDS[sid])["positive"]
    nopeak = inn.place_fields(BOUNDS[sid], peak=None)["positive"]
    rule = "peak > 100% dF/F" if peak.sum() >= 5 else "no peak rule"
    sel = peak if peak.sum() >= 5 else nopeak
    SEL[sid] = sel
    X = s.dff[RUN[sid]][:, m]
    laps = B[sid]["lap"][RUN[sid]]
    Xl, y, g = inn.lap_tuning(X, BINS[sid], laps)
    err = lambda cols: inn.decode_position(Xl[:, cols], y, g, B[sid]["length"])
    e_all, e_sel = err(np.arange(m.sum())), err(np.flatnonzero(sel))
    e_rand = [err(rng.choice(m.sum(), sel.sum(), replace=False)) for _ in range(10)]
    e_shuf = []
    for _ in range(3):
        Xs = np.roll(X[:, sel], rng.integers(int(60 * clock[sid]), len(X) - int(60 * clock[sid])), axis=0)
        e_shuf.append(inn.decode_position(*inn.lap_tuning(Xs, BINS[sid], laps), B[sid]["length"]))
    if peak.sum() >= 1:
        e_peak = err(np.flatnonzero(peak))
    else:
        e_peak = np.nan
    rows.append({"session": sid, "laps": len(np.unique(g)), "rule": rule, "ROIs": m.sum(), "selected": sel.sum(),
                 "selected with peak rule": peak.sum(), "error all (cm)": e_all, "error selected": e_sel,
                 "error peak-rule cells": e_peak, "error random, mean": np.mean(e_rand),
                 "error random, min": np.min(e_rand), "error shuffled, mean": np.mean(e_shuf),
                 "chance (cm)": B[sid]["length"] * 100 / 4})
dec = pd.DataFrame(rows).set_index("session")
dec["selected / all"] = dec["error selected"] / dec["error all (cm)"]
dec["holds"] = (dec["selected / all"] <= 1.25) & (dec["error selected"] < dec["error random, mean"]) & \
               (dec["error selected"] < dec["error shuffled, mean"])
print(dec.T.to_string())
print("Chance is the mean error of a uniform guess around a loop, a quarter of the belt.")

**Result: not reproduced.** The criterion holds in one session of three. In 21324 the 162
selected ROIs decode with an error of 13.8 cm, against 13.2 cm from all ROIs, 14.4 cm from random
subsets and 48.7 cm with position shuffled. In 21333 the selected ROIs do no better than random
subsets (25.2 against 24.8 cm). In 21322 the 6 ROIs of the peak rule give twice the error of all
ROIs (30.3 against 15.4 cm). In every session shuffling position brings the error to about chance
(49 to 53 cm, chance 52.7 cm), and all ROIs decode position well (13 to 21 cm).

## Selectivity index (`fig4f-selectivity-no-difference`)

> The selectivity index of spatially selective cells did not differ between subtypes (one-way
> ANOVA, p = 0.35).

The cells are those selected for the decoding above. `fig4e-bic-recurrence` is blocked: the
dandiset has 3 BiC.

In [ ]:
si = np.full((len(order), len(s0.roi_ids)), np.nan)
for k, sid in enumerate(order):
    m = np.flatnonzero(IMAGED[sid])
    idx = m[SEL[sid]]
    si[k, idx] = inn.sparsity_index(BOUNDS[sid]["tc"][SEL[sid]])
sel_cells = pd.DataFrame({"subtype": s0.subtype, "si": np.nanmean(si, 0), "times selected": (~np.isnan(si)).sum(0)})
sel_cells = sel_cells[(sel_cells.subtype != "") & (sel_cells["times selected"] > 0)]
print(sel_cells.groupby("subtype").si.agg(["count", "mean", "median"]).reindex(inn.ORDER).to_string())
use = [t for t in inn.ORDER if (sel_cells.subtype == t).sum() >= 3]
F, p = stats.f_oneway(*[sel_cells.si[sel_cells.subtype == t] for t in use])
print(f"\nOne-way ANOVA over {use}: F = {F:.2f}, p = {p:.2g}")
print(f"Effect size: subtype means range from {sel_cells.groupby('subtype').si.mean()[use].min():.3f} "
      f"to {sel_cells.groupby('subtype').si.mean()[use].max():.3f}; overall SD {sel_cells.si.std():.3f}")

**Result: reproduced.** The one-way ANOVA across the six subtypes gives p = 0.17, so no
difference is found. Subtype means range from 0.012 to 0.033, with a standard deviation of 0.020
across cells. The indices are small because dF/F here has a baseline near 0.12 in every bin.

## Stability of spatially selective PVBC and SomC (`fig4h-pvbc-more-stable`)

> Spatially selective PVBC had a more stable within-session recruitment than spatially selective
> SomC.

Stability is the correlation between the tuning curves of odd and even laps.

In [ ]:
OE = {sid: np.full(len(s0.roi_ids), np.nan) for sid in order}
for sid in order:
    m = IMAGED[sid]
    OE[sid][m] = inn.odd_even_correlation(sessions[sid].dff[RUN[sid]][:, m], BINS[sid], B[sid]["lap"][RUN[sid]])
oe_sel = np.full((len(order), len(s0.roi_ids)), np.nan)
for k, sid in enumerate(order):
    idx = np.flatnonzero(IMAGED[sid])[SEL[sid]]
    oe_sel[k, idx] = OE[sid][idx]
per_cell = pd.DataFrame({"subtype": s0.subtype, "r": np.nanmean(oe_sel, 0)}).dropna()
spatial_pv_som = {t: per_cell.r[per_cell.subtype == t].to_numpy() for t in ("PVBC", "SomC")}
print({t: len(v) for t, v in spatial_pv_som.items()}, "spatially selective cells (odd-even r averaged over the sessions in which each was selected)")
if min(len(v) for v in spatial_pv_som.values()) >= 5:
    a, b = spatial_pv_som["PVBC"], spatial_pv_som["SomC"]
    print(f"PVBC mean {a.mean():.3f}, SomC mean {b.mean():.3f}, difference {a.mean() - b.mean():.3f}, "
          f"unpaired t-test p = {stats.ttest_ind(a, b).pvalue:.2g}")
    for sid in order:
        idx = np.flatnonzero(IMAGED[sid])[SEL[sid]]
        v = {t: OE[sid][idx][s0.subtype[idx] == t] for t in ("PVBC", "SomC")}
        print(f"  session {sid}: PVBC {v['PVBC'].mean():.3f} (n = {len(v['PVBC'])}), SomC {v['SomC'].mean():.3f} (n = {len(v['SomC'])})")

**Result: not reproduced.** 22 spatially selective PVBC and 9 SomC qualify. PVBC are less stable
than SomC, not more (mean odd-even correlation 0.678 against 0.736, unpaired t-test p = 0.42), and
the same holds in each session with both subtypes present.

## Within-session stability by subtype (`s6-within-session-order`)

> Within-session stability of tuning curves differed between subtypes, PVBC showing the highest
> and CCKC the lowest odd-even correlations.

All imaged ROIs of the six subtypes, regardless of spatial selectivity.

In [ ]:
oe = pd.DataFrame({"subtype": s0.subtype, **{sid: OE[sid] for sid in order}})
oe["mean"] = oe[order].mean(axis=1)
oe = oe[(oe.subtype != "") & oe["mean"].notna()]
table = oe.groupby("subtype")[["mean", *order]].mean().reindex(inn.ORDER)
table.insert(0, "n", oe.groupby("subtype").size().reindex(inn.ORDER))
print(table.to_string())
eligible = table[table.n >= 5]
print(f"\nAmong subtypes with at least 5 cells: highest {eligible['mean'].idxmax()}, lowest {eligible['mean'].idxmin()}")
F, p = stats.f_oneway(*[oe["mean"][oe.subtype == t] for t in eligible.index])
print(f"One-way ANOVA across them: F = {F:.2f}, p = {p:.2g}")

**Result: partially reproduced.** CCKC have the lowest mean odd-even correlation (0.500). PVBC
(0.598) are second to SomC (0.606), not first. The ANOVA across the five subtypes with at least 5
cells gives p = 0.56, so the subtypes do not differ significantly here.

## Stability across sessions (`s6-later-lower-above-chance`)

> Tuning curve correlations of the same cells after 1 hour and after 24 hours were lower than
> within-session correlations, and still higher than chance.

The same-day pair (21324 then 21322, starts 2.5 h apart) stands in for 1 hour, and the two pairs
with 21333 (54 and 56 h) for 24 hours. The ROIs in the six subtypes imaged in both sessions.

In [ ]:
rng = rng_for("s6-later-lower-above-chance")
rows = []
for a, b in [(order[1], order[2]), (order[0], order[1]), (order[0], order[2])]:
    m = IMAGED[a] & IMAGED[b] & (s0.subtype != "")
    across = inn.rowwise_corr(TC[a][m], TC[b][m])
    within = OE[a][m]
    ok = ~np.isnan(across) & ~np.isnan(within)
    obs = across[ok].mean()
    idx, types = np.flatnonzero(m)[ok], s0.subtype[m][ok]
    null_all, null_type = [], []
    for _ in range(1000):
        null_all.append(np.nanmean(inn.rowwise_corr(TC[a][idx], TC[b][rng.permutation(idx)])))
        perm = idx.copy()
        for t in np.unique(types):
            perm[types == t] = rng.permutation(idx[types == t])
        null_type.append(np.nanmean(inn.rowwise_corr(TC[a][idx], TC[b][perm])))
    gap_h = (pd.Timestamp(sessions[b].start_time) - pd.Timestamp(sessions[a].start_time)).total_seconds() / 3600
    rows.append({"pair": f"{a}-{b}", "hours between starts": gap_h, "cells": ok.sum(),
                 "within-session r (earlier)": within[ok].mean(), "across r": obs,
                 "Wilcoxon p (across < within)": stats.wilcoxon(across[ok], within[ok], alternative="less").pvalue,
                 "null mean, all shuffled": np.mean(null_all),
                 "p, all shuffled": (1 + np.sum(np.array(null_all) >= obs)) / 1001,
                 "null mean, within subtype": np.mean(null_type),
                 "p, within subtype": (1 + np.sum(np.array(null_type) >= obs)) / 1001})
print(pd.DataFrame(rows).set_index("pair").T.to_string())

**Result: partially reproduced.** Correlations across sessions are lower than within the earlier
session in every pair (0.22 against 0.61 for the same-day pair, Wilcoxon p = 4e-07; 0.06 and 0.14
for the two-day pairs). Above chance holds only for the same-day pair with identities shuffled
among all cells (p = 0.047), not with shuffles within subtype (p = 0.10), and for neither two-day
pair (p = 0.075 and 0.53 with all cells shuffled). As cell identity across sessions is not
established, this may reflect the ROIs as much as the cells.

## Summary

| Claim | Outcome | Note |
|---|---|---|
| `fig1-cells-per-mouse` | Partially reproduced | 231 ROIs, in range; 74 or 83 identified, below 92.6 to 149.4 |
| `fig1-depth-distribution` | Blocked: data missing | No cell depth or coordinates in the files |
| `fig2c-most-positive` | Reproduced | 90.5% positive, 0.095 negative among the six subtypes |
| `fig2c-cck-negative` | Reproduced | 6 of 15 CCKC negative against 1 of 59 others |
| `fig2c-subtypes-differ` | Reproduced | ANOVA p = 0.0015, driven by CCKC |
| `fig2c-ngfc-less-modulated` | Not reproduced | IvC/NGFC 0.511 against 0.484, p = 0.58 |
| `fig2d-negative-cells-in-oriens` | Blocked: data missing | No cell depth |
| `s2-lags` | Not reproduced | Negatively correlated ROIs not more tightly locked |
| `s2-cck-run-stop` | Partially reproduced | CCKC flat at run stops while others drop; no rise on average |
| `s3-subtype-classification` | Blocked: data missing | One mouse of the training set, no depth |
| `fig3-swr-pvbc-vs-somc`, `fig3-swr-bimodal` | Blocked: data missing | No LFP or 40 Hz sessions |
| `fig4-fraction-spatial` | Not reproduced | 1.0% with the peak rule, 69.8% without |
| `fig4-fraction-negative` | Not reproduced | 66.4% |
| `fig4d-decoding` | Not reproduced | Criterion holds in 1 session of 3 |
| `fig4e-bic-recurrence` | Blocked: other | 3 BiC |
| `fig4f-selectivity-no-difference` | Reproduced | ANOVA p = 0.17 |
| `fig4h-pvbc-more-stable` | Not reproduced | PVBC 0.678, SomC 0.736 |
| `s6-within-session-order` | Partially reproduced | CCKC lowest; PVBC second to SomC; ANOVA p = 0.56 |
| `s6-later-lower-above-chance` | Partially reproduced | Lower in every pair; above chance only same-day, one reading |
| `fig5-context-decorrelation`, `fig5-novel-context-decrease` | Blocked: data missing | No novel-context sessions |
| `fig6-reward-somc-bic`, `fig6-old-zone` | Blocked: data missing | No goal-oriented sessions or licking |
| `fig7-vip-subtypes-immobility`, `fig7-isi-correlations` | Blocked: data missing | VIP-Cre and strategy 3 mice not included |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How this differs from the paper's analysis:

- One mouse with 74 identified interneurons, where the paper pools 6 or 7 mice per analysis.
- The frame rates are fitted from the data for two sessions, as the stated rate does not fit.
- The ROI with the same index is taken as the same cell in every session; the data are consistent
  with this but do not show it.
- The 1 hour and 24 hour comparisons use sessions 2.5 h and about two days apart.
- Choices the methods leave open: velocity smoothed over 0.5 s, run events at least 5 s apart,
  place fields allowed to wrap around the belt, a one-vs-one linear SVM, shuffled position made by
  rotating dF/F against position.

What was found about the dandiset:

- The stated imaging rate, 5.7981 Hz, is wrong for sessions 21324 and 21322, whose frames fit
  about 4.39 Hz; on the stated clock their activity does not follow running.
- The dF/F unit is given as lumens; the Fluorescence series is a link to the dF/F series, so raw
  fluorescence is missing; the subtype column's description is a placeholder ("Recording
  location in Middle Earth"), and the table has one row more than there are ROIs.
- The labels "NOTHING" and "UNIDENTIFIED" are not explained, nor are the reward and water
  intervals; there is no cell depth, no mapping of ROIs across sessions, and no lick record.
- All three sessions are called "session 1 of 4", and the fourth session is not included.
- The dandiset holds one of the paper's 16 mice and only its random-foraging sessions, without
  saying which part of the paper it covers.
- With the paper's place-field rule and a peak above 100% dF/F, almost no ROI qualifies, as only
  1.1% of tuning curves reach 1.0 dF/F; the dF/F here may be scaled differently from the paper's.